# TransLuxPop / CivitasGrid-TLP: Multi-Baselines + Affine Calibration (RMSE / CL)

这个 Notebook 用于跑论文主实验（**基线模型 + 校准层**）：

- **Baselines（7 个）**：XGBoost、Ridge、ElasticNet、SVR、RandomForest、ExtraTrees、MLP  
- **每个 baseline 的 3 组**：
  1. **Base**：只训练 baseline（标准回归损失）
  2. **CL-only**：固定 baseline，用 **CL** 训练一个仿射校准层（$\hat{y}' = a\hat{y}+b$）
  3. **Mix / RMSE\_CL**：固定 baseline，用 **RMSE + CL** 训练仿射校准层

- **Splits**：RandomSplit、GroupSplit(grid\_id)  
- **Targets**：dWorldPop、dVIIRS

输出：
- `results_tail_tables.xlsx`：论文表格用的汇总结果
- `artifacts/`：可视化与诊断图（按你的原本结构）

> 想临时去掉某个模型：改 `BASELINE_NAMES` / `MODEL_NAMES` 列表即可。

In [1]:
# 导入依赖 + 固定随机种子（可复现）
import os
import random
import warnings
from dataclasses import dataclass
from typing import Dict, Tuple, Optional, List

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import Ridge, ElasticNet
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.neural_network import MLPRegressor

import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

SEED = 42

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(SEED)

print('Versions:')
import sklearn
from xgboost import XGBRegressor
import xgboost
print('  pandas:', pd.__version__)
print('  numpy:', np.__version__)
print('  sklearn:', sklearn.__version__)
print('  xgboost:', xgboost.__version__)


Versions:
  pandas: 2.3.3
  numpy: 2.3.5
  sklearn: 1.8.0
  xgboost: 3.1.2


In [2]:
# 读取并初步检查 CivitasGrid-TLP / TransLuxPop（HQ）数据

data_path = 'grids_set_4_HQ.xlsx'
if not os.path.exists(data_path):
    alt = '/mnt/data/grids_set_4_HQ.xlsx'
    if os.path.exists(alt):
        data_path = alt

assert os.path.exists(data_path), f'找不到数据文件：{data_path}（请确认已上传）'

df = pd.read_excel(data_path)
df.columns = df.columns.str.strip()

print('Shape:', df.shape)
print('Has grid_id/year:', {'grid_id','year'}.issubset(df.columns))
print('Targets exist:', {'dVIIRS','dWorldPop'}.issubset(df.columns))

display(df.head(3))

# 为避免部分树模型在重复值上出现不稳定的分裂（可选），对 _use 与 last_year 特征加入极小扰动
use_cols = [c for c in df.columns if c.endswith('_use')] + ["VIIRS_last_year", "WorldPop_last_year"]
use_cols = [c for c in use_cols if c in df.columns]
if len(use_cols) > 0:
    df[use_cols] = df[use_cols] + np.random.uniform(-0.01, 0.01, size=df[use_cols].shape)
    print('Applied tiny jitter to:', use_cols[:6], '...')


Shape: (28330, 37)
Has grid_id/year: True
Targets exist: True


,grid_id,year,nation_code,lat_min,lon_min,lat_max,lon_max,cell_area,region_type,city_type,...,City,description,Intersec_use,mot_use,tru_use,pri_use,sec_use,ter_use,urb_use,len_unc
0,1000,2015.0,USA,37.556543,-77.420981,37.572814,-77.40471,2.596018,commercial,developing,...,Richmond,发展中城市：市中心商业核心，路网密集/夜光强。,108.242695,2.599565,0.850466,0.047647,1.75934,0.545176,5.066507,0
1,1000,2016.0,USA,37.556543,-77.420981,37.572814,-77.40471,2.596018,commercial,developing,...,Richmond,发展中城市：市中心商业核心，路网密集/夜光强。,108.242695,2.599565,0.850466,0.047647,1.75934,0.545176,5.066507,0
2,1000,2017.0,USA,37.556543,-77.420981,37.572814,-77.40471,2.596018,commercial,developing,...,Richmond,发展中城市：市中心商业核心，路网密集/夜光强。,108.242695,2.599565,0.850466,0.047647,1.75934,0.545176,5.066507,0


Applied tiny jitter to: ['Intersec_use', 'mot_use', 'tru_use', 'pri_use', 'sec_use', 'ter_use'] ...


In [3]:
# 构造 covid_intensity 特征，并定义特征/目标列

covid_map = {
    2015: 0.0,
    2016: 0.0,
    2017: 0.0,
    2018: 0.0,
    2019: 0.05,
    2020: 0.8,
    2021: 1.0,
    2022: 0.6,
    2023: 0.4,
    2024: 0.2,
}

# year 极少缺失：无法构造 covid_intensity → 丢弃（可复现）
df = df.dropna(subset=['year']).copy()
df['year'] = df['year'].astype(int)
df['covid_intensity'] = df['year'].map(covid_map).fillna(0.0)

feature_cols = [
    'mot_use', 'tru_use', 'pri_use', 'sec_use', 'ter_use', 'urb_use',
    'VIIRS_last_year', 'WorldPop_last_year', 'covid_intensity',
    'region_type', 'city_type'
]

target_cols = ['dVIIRS', 'dWorldPop']

missing_cols = [c for c in feature_cols + target_cols + ['grid_id','year'] if c not in df.columns]
assert len(missing_cols) == 0, f'缺少列：{missing_cols}'

print('Feature cols:', feature_cols)
print('Targets:', target_cols)
print('Unique region_type:', df['region_type'].nunique(), 'Unique city_type:', df['city_type'].nunique())


Feature cols: ['mot_use', 'tru_use', 'pri_use', 'sec_use', 'ter_use', 'urb_use', 'VIIRS_last_year', 'WorldPop_last_year', 'covid_intensity', 'region_type', 'city_type']
Targets: ['dVIIRS', 'dWorldPop']
Unique region_type: 5 Unique city_type: 5


In [4]:
# 缺失值处理策略与缺失概况
# - 数值特征：中位数填补（仅在训练集 fit，防泄漏）
# - 类别特征：缺失填 'Unknown'（仅在训练集 fit，防泄漏）
# - 目标 y：建模时必须非缺失（每个目标分别过滤）

numeric_cols = [
    'mot_use','tru_use','pri_use','sec_use','ter_use','urb_use',
    'VIIRS_last_year','WorldPop_last_year','covid_intensity'
]

categorical_cols = ['region_type','city_type']

missing_summary = df[feature_cols + target_cols].isna().mean().sort_values(ascending=False)
display(missing_summary.to_frame('missing_ratio').head(12))


,missing_ratio
VIIRS_last_year,0.100004
dWorldPop,0.100004
dVIIRS,0.100004
WorldPop_last_year,0.100004
mot_use,0.000000
ter_use,0.000000
sec_use,0.000000
pri_use,0.000000
tru_use,0.000000
covid_intensity,0.000000


In [5]:
# 定义预处理器（数值缺失填补 + 标准化 + 类别 One-Hot）

def build_preprocessor(numeric_features: List[str], categorical_features: List[str]) -> ColumnTransformer:
    try:
        ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)

    numeric_tf = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    categorical_tf = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
        ('onehot', ohe)
    ])

    pre = ColumnTransformer(
        transformers=[
            ('num', numeric_tf, numeric_features),
            ('cat', categorical_tf, categorical_features)
        ],
        remainder='drop',
        verbose_feature_names_out=False
    )
    return pre

preprocessor = build_preprocessor(numeric_cols, categorical_cols)


In [7]:
# Contrastive Learning 目标函数 + 混合训练（RMSE + CL）+ 指标

# -------------------------
# Contrastive config knobs
# -------------------------
CL_BATCH_SIZE = 128
CL_EPOCHS = 40
CL_LR = 0.05
CL_TEMP_FACTOR = 1.0
CL_EPS = 1e-12


def compute_temperature(y_train: np.ndarray, factor: float = CL_TEMP_FACTOR) -> float:
    s = float(np.std(y_train))
    return max(s * float(factor), 1e-3)


def contrastive_loss_and_p(y_pred: np.ndarray, y_true: np.ndarray, T: float):
    # batch 内全对比：negatives = batch 中其它 y_j
    # p_i = exp(-|yhat_i - y_i|/T) / sum_j exp(-|yhat_i - y_j|/T)
    y_pred = np.asarray(y_pred, dtype=float)
    y_true = np.asarray(y_true, dtype=float)
    B = len(y_true)
    if B == 0:
        return np.nan, np.array([])
    T = float(max(T, 1e-6))

    D = np.abs(y_pred[:, None] - y_true[None, :])  # (B,B)
    logits = -D / T
    # 稳定 softmax
    logits = logits - np.max(logits, axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    denom = np.sum(exp_logits, axis=1)
    numer = np.diag(exp_logits)
    p = numer / np.maximum(denom, CL_EPS)
    loss = float(np.mean(-np.log(np.maximum(p, CL_EPS))))
    return loss, p


def contrastive_grad_yhat(y_pred: np.ndarray, y_true: np.ndarray, T: float) -> np.ndarray:
    # 计算 dL_con / d y_pred（batch 全对比，L1 距离）
    y_pred = np.asarray(y_pred, dtype=float)
    y_true = np.asarray(y_true, dtype=float)
    B = len(y_true)
    if B == 0:
        return np.array([])
    T = float(max(T, 1e-6))

    D = np.abs(y_pred[:, None] - y_true[None, :])
    logits = -D / T
    logits = logits - np.max(logits, axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    denom = np.sum(exp_logits, axis=1, keepdims=True)
    P = exp_logits / np.maximum(denom, CL_EPS)  # row softmax

    # sign matrix s_ij = sign(yhat_i - y_j)
    S = np.sign(y_pred[:, None] - y_true[None, :])
    S_diag = np.diag(S)

    # grad_i = (1/T) * ( s_ii - sum_j P_ij * s_ij )
    grad = (S_diag - np.sum(P * S, axis=1)) / T
    grad = grad / float(B)  # mean over batch
    return grad


def rmse_loss_and_grad_yhat(y_pred: np.ndarray, y_true: np.ndarray) -> tuple:
    y_pred = np.asarray(y_pred, dtype=float)
    y_true = np.asarray(y_true, dtype=float)
    B = len(y_true)
    if B == 0:
        return np.nan, np.array([])
    err = y_pred - y_true
    mse = float(np.mean(err * err))
    rmse = float(np.sqrt(max(mse, 0.0)))
    denom = max(rmse, 1e-6)
    grad = (err / denom) / float(B)
    return rmse, grad


@dataclass
class CalibResult:
    a: float
    b: float
    logs: Dict[str, List[float]]


def train_affine_calibrator(
    yhat_base: np.ndarray,
    y_true: np.ndarray,
    T: float,
    mode: str = 'cl',
    rmse_weight: float = 0.95,
    cl_weight: float = 0.05,
    epochs: int = CL_EPOCHS,
    batch_size: int = CL_BATCH_SIZE,
    lr: float = CL_LR,
    seed: int = SEED,
) -> CalibResult:
    """
    训练一个非常轻量的可微校准层： y' = a*yhat + b
    - (Baseline)_CL：优化 L_con
    - (Baseline)_RMSE_CL：优化 0.95*RMSE_norm + 0.05*CL_norm

    ✅ 改动：
    1) cl0（baseline contrastive loss 用于归一化）改成子采样估计，避免全量 O(N^2) 爆炸。
    2) 增加 flush 的心跳输出，方便判断是否"活着"。
    """

    import sys
    import time
    import numpy as np

    t_start = time.time()
    rng = np.random.default_rng(seed)
    yhat_base = np.asarray(yhat_base, dtype=float)
    y_true = np.asarray(y_true, dtype=float)

    n = len(y_true)
    if n == 0:
        return CalibResult(a=1.0, b=0.0, logs={'loss': [], 'rmse': [], 'cl': []})

    # -----------------------------
    # ✅ 关键：CL 的 O(N^2) 防爆策略
    # -----------------------------
    # 这些阈值你可以按机器能力调：
    # - 太大：容易慢/爆内存
    # - 太小：cl0 估计噪声大，但通常也够用了（反正是归一化基准）
    CL0_MAX_N = 3000          # 用于估计 cl0 的最大样本数
    CL0_BATCH = 512           # cl0 用的小 batch（避免 contrastive 里构造大矩阵）
    CL_EVAL_MAX_N = 3000      # epoch end 记录 cl_full 时同样子采样，避免日志阶段卡死

    def _subsample(x: np.ndarray, y: np.ndarray, max_n: int):
        if len(y) <= max_n:
            return x, y
        idx = rng.choice(len(y), size=max_n, replace=False)
        return x[idx], y[idx]

    def _estimate_cl_mean(x: np.ndarray, y: np.ndarray, T: float, bs: int) -> float:
        """
        用小 batch 平均来估计 contrastive loss，避免一次性对全量做 O(N^2)。
        注意：这是"估计"，不是严格等价全量 loss，但用来当归一化标尺非常够用。
        """
        n_local = len(y)
        if n_local == 0:
            return 0.0
        perm = rng.permutation(n_local)
        vals = []
        for s in range(0, n_local, bs):
            bidx = perm[s:s + bs]
            # 这里调用你原来的 contrastive loss（假设它在 batch 内部可算）
            cl_val, _ = contrastive_loss_and_p(x[bidx], y[bidx], T)
            vals.append(float(cl_val))
        return float(np.mean(vals)) if len(vals) else 0.0

    # -----------------------------
    # 参数初始化
    # -----------------------------
    a, b = 1.0, 0.0

    # baseline rmse 用于归一化（这一步是 O(N) 安全的）
    rmse0, _ = rmse_loss_and_grad_yhat(a * yhat_base + b, y_true)
    rmse0 = float(max(rmse0, 1e-6))

    # ✅ baseline cl 用于归一化：子采样 + 小 batch 估计，防止全量 O(N^2)
    if mode in ('cl', 'rmse_cl'):
        xb0, yb0 = _subsample(yhat_base, y_true, CL0_MAX_N)
        cl0 = _estimate_cl_mean(a * xb0 + b, yb0, T, CL0_BATCH)
        cl0 = float(max(cl0, 1e-6))
    else:
        cl0 = 1.0  # 理论上不会走到这，因为 mode 会校验

    print(
        f"[calib] mode={mode} n={n} epochs={int(epochs)} batch={int(batch_size)} lr={lr} "
        f"| rmse0={rmse0:.6f} cl0≈{cl0:.6f} (subsampled)",
        flush=True
    )
    sys.stdout.flush()

    # Adam
    m_a = v_a = 0.0
    m_b = v_b = 0.0
    beta1, beta2 = 0.9, 0.999
    eps = 1e-8

    logs = {'loss': [], 'rmse': [], 'cl': []}

    # 心跳频率：每多少 step 打印一次（你可以调小让输出更多）
    HEARTBEAT_EVERY = 50

    step = 0
    for ep in range(int(epochs)):
        idx = rng.permutation(n)
        ep_loss_acc = 0.0
        ep_rmse_acc = 0.0
        ep_cl_acc = 0.0
        nb = 0

        for start in range(0, n, int(batch_size)):
            step += 1
            batch = idx[start:start + int(batch_size)]
            yb = y_true[batch]
            xb = yhat_base[batch]

            yhat = a * xb + b

            # losses + grads wrt yhat
            rmse, grad_rmse = rmse_loss_and_grad_yhat(yhat, yb)
            cl, _ = contrastive_loss_and_p(yhat, yb, T)
            grad_cl = contrastive_grad_yhat(yhat, yb, T)

            if mode == 'cl':
                loss = cl
                grad_yhat = grad_cl
            elif mode == 'rmse_cl':
                rmse_norm = rmse / rmse0
                cl_norm = cl / cl0
                loss = rmse_weight * rmse_norm + cl_weight * cl_norm
                grad_yhat = rmse_weight * (grad_rmse / rmse0) + cl_weight * (grad_cl / cl0)
            else:
                raise ValueError('mode must be cl or rmse_cl')

            # chain rule to a,b
            g_a = float(np.sum(grad_yhat * xb))
            g_b = float(np.sum(grad_yhat))

            # Adam update
            m_a = beta1 * m_a + (1 - beta1) * g_a
            v_a = beta2 * v_a + (1 - beta2) * (g_a * g_a)
            m_b = beta1 * m_b + (1 - beta1) * g_b
            v_b = beta2 * v_b + (1 - beta2) * (g_b * g_b)

            m_a_hat = m_a / (1 - beta1 ** step)
            v_a_hat = v_a / (1 - beta2 ** step)
            m_b_hat = m_b / (1 - beta1 ** step)
            v_b_hat = v_b / (1 - beta2 ** step)

            a -= lr * m_a_hat / (np.sqrt(v_a_hat) + eps)
            b -= lr * m_b_hat / (np.sqrt(v_b_hat) + eps)

            # 记录 batch 统计（用于更"活着"的输出）
            ep_loss_acc += float(loss)
            ep_rmse_acc += float(rmse)
            ep_cl_acc += float(cl)
            nb += 1

            if (step % HEARTBEAT_EVERY) == 0:
                avg_loss = ep_loss_acc / max(nb, 1)
                avg_rmse = ep_rmse_acc / max(nb, 1)
                avg_cl = ep_cl_acc / max(nb, 1)
                print(
                    f"[calib] ep={ep+1}/{int(epochs)} step={step} "
                    f"| loss≈{avg_loss:.6f} rmse≈{avg_rmse:.6f} cl≈{avg_cl:.6f} "
                    f"| a={a:.5f} b={b:.5f} | {time.time()-t_start:.1f}s",
                    flush=True
                )
                sys.stdout.flush()

        # -----------------------------
        # epoch end logging（⚠️ 原版这里会全量算 cl_full -> 可能又 O(N^2) 卡死）
        # 改成：rmse 全量（安全） + cl 子采样估计（安全）
        # -----------------------------
        yhat_full = a * yhat_base + b
        rmse_full, _ = rmse_loss_and_grad_yhat(yhat_full, y_true)

        # cl_full 只用子采样估计
        xbe, ybe = _subsample(yhat_full, y_true, CL_EVAL_MAX_N)
        cl_full = _estimate_cl_mean(xbe, ybe, T, CL0_BATCH)

        if mode == 'rmse_cl':
            loss_full = rmse_weight * (rmse_full / rmse0) + cl_weight * (cl_full / cl0)
        else:
            loss_full = cl_full

        logs['loss'].append(float(loss_full))
        logs['rmse'].append(float(rmse_full))
        logs['cl'].append(float(cl_full))

        print(
            f"[calib] ✅ epoch_end ep={ep+1}/{int(epochs)} "
            f"| loss={loss_full:.6f} rmse={float(rmse_full):.6f} cl≈{float(cl_full):.6f} "
            f"| a={a:.5f} b={b:.5f} | {time.time()-t_start:.1f}s",
            flush=True
        )
        sys.stdout.flush()

    return CalibResult(a=float(a), b=float(b), logs=logs)


def compute_tail_thresholds(y_train: np.ndarray, q_low: float = 0.05, q_high: float = 0.95) -> Tuple[float, float]:
    q05 = np.quantile(y_train, q_low)
    q95 = np.quantile(y_train, q_high)
    return float(q05), float(q95)


def tail_mask(y: np.ndarray, q05: float, q95: float) -> np.ndarray:
    y = np.asarray(y)
    return (y <= q05) | (y >= q95)


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = r2_score(y_true, y_pred)
    abs_err = np.abs(y_true - y_pred)
    return {
        'MAE': float(mae),
        'RMSE': float(rmse),
        'R2': float(r2),
        'AbsErr_P95': float(np.quantile(abs_err, 0.95)),
        'AbsErr_P99': float(np.quantile(abs_err, 0.99)),
    }


def evaluate_with_tail(y_true: np.ndarray, y_pred: np.ndarray, q05: float, q95: float) -> Dict[str, float]:
    out = {}
    out.update({f'All_{k}': v for k, v in regression_metrics(y_true, y_pred).items()})
    m = tail_mask(y_true, q05, q95)
    out['Tail_Rate'] = float(m.mean())
    if m.sum() > 0:
        out.update({f'Tail_{k}': v for k, v in regression_metrics(y_true[m], y_pred[m]).items()})
    else:
        out.update({f'Tail_{k}': np.nan for k in ['MAE','RMSE','R2','AbsErr_P95','AbsErr_P99']})
    return out

In [8]:
# 定义 Baseline 模型与超参（显式写出，便于复现）

BASELINES = {
'XGBoost': dict(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    min_child_weight=1.0,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=SEED,
    n_jobs=-1,
    verbosity=0,
),
    'Ridge': dict(alpha=1.0),
    'ElasticNet': dict(alpha=1.0, l1_ratio=0.5, max_iter=5000, random_state=SEED),
    'SVR': dict(C=10.0, epsilon=0.1, kernel='rbf'),
    'RandomForest': dict(n_estimators=150, max_depth=None, min_samples_split=2, min_samples_leaf=1, random_state=SEED, n_jobs=-1),
    'ExtraTrees': dict(n_estimators=300, max_depth=None, min_samples_split=2, min_samples_leaf=1, random_state=SEED, n_jobs=-1),
    'MLP': dict(hidden_layer_sizes=(128, 64), activation='relu', solver='adam', alpha=1e-4, max_iter=400, random_state=SEED, verbose=False, early_stopping=True, validation_fraction=0.1, n_iter_no_change=15),
}

ALIASES = {'RF': 'RandomForest', 'ETR': 'ExtraTrees', 'XGB': 'XGBoost'}

def build_model(name: str):
    name = ALIASES.get(name, name)
    cfg = BASELINES[name]
    if name == 'Ridge':
        return Ridge(**cfg)
    if name == 'ElasticNet':
        return ElasticNet(**cfg)
    if name == 'SVR':
        return SVR(**cfg)
    if name == 'RandomForest':
        return RandomForestRegressor(**cfg)
    if name == 'ExtraTrees':
        return ExtraTreesRegressor(**cfg)

    if name == 'XGBoost':
        return XGBRegressor(**cfg)
    if name == 'MLP':
        return MLPRegressor(**cfg)
    raise KeyError(name)

BASELINE_NAMES = list(BASELINES.keys())
print('Baselines:', BASELINE_NAMES)


Baselines: ['XGBoost', 'Ridge', 'ElasticNet', 'SVR', 'RandomForest', 'ExtraTrees', 'MLP']


In [9]:
# 数据切分：Random / GroupSplit(grid_id)

def make_random_split(df_in: pd.DataFrame, test_size=0.15, val_size=0.15, seed=SEED):
    idx = np.arange(len(df_in))
    train_idx, temp_idx = train_test_split(idx, test_size=test_size+val_size, random_state=seed, shuffle=True)
    rel_test = test_size / (test_size + val_size)
    val_idx, test_idx = train_test_split(temp_idx, test_size=rel_test, random_state=seed, shuffle=True)
    return train_idx, val_idx, test_idx


def make_group_split_by_grid(df_in: pd.DataFrame, group_col='grid_id', test_size=0.15, val_size=0.15, seed=SEED):
    groups = df_in[group_col].astype(str).values

    # 先划 test
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    train_val_idx, test_idx = next(gss.split(np.arange(len(df_in)), groups=groups))

    # 再从 train_val 中划 val
    groups_tv = groups[train_val_idx]
    gss2 = GroupShuffleSplit(n_splits=1, test_size=val_size/(1.0 - test_size), random_state=seed)
    train_idx_rel, val_idx_rel = next(gss2.split(train_val_idx, groups=groups_tv))
    train_idx = train_val_idx[train_idx_rel]
    val_idx = train_val_idx[val_idx_rel]

    return train_idx, val_idx, test_idx


In [10]:
import time
# 训练与推理：每个 baseline -> 3 组（Base / _CL / _RMSE_CL）

@dataclass
class FitArtifacts:
    y_true: np.ndarray
    y_pred: np.ndarray
    tail_q05: float
    tail_q95: float
    diag_test: Optional[np.ndarray] = None
    diag_name: Optional[str] = None
    extra: Optional[dict] = None


def build_pipeline(model_name: str) -> Pipeline:
    model = build_model(model_name)
    pipe = Pipeline(steps=[
        ('pre', preprocessor),
        ('model', model)
    ])
    return pipe


def _filter_nonmissing(df_in: pd.DataFrame, idx: np.ndarray, target: str) -> np.ndarray:
    sub = df_in.iloc[idx]
    m = sub[target].notna().values
    return idx[m]


def _approx_test_diag_p(y_pred: np.ndarray, y_true: np.ndarray, T: float, batch_size: int = 256, seed: int = SEED) -> np.ndarray:
    # 诊断用：在 test 上用随机 mini-batch 计算 p_i（避免 O(n^2) 全量）
    rng = np.random.default_rng(seed)
    n = len(y_true)
    if n == 0:
        return np.array([])
    order = rng.permutation(n)
    ps = []
    for start in range(0, n, batch_size):
        b = order[start:start+batch_size]
        _, p = contrastive_loss_and_p(y_pred[b], y_true[b], T)
        ps.append(p)
    return np.concatenate(ps, axis=0)


def fit_one_model_three_variants(
    df_in: pd.DataFrame,
    train_idx: np.ndarray,
    val_idx: np.ndarray,
    test_idx: np.ndarray,
    target: str,
    baseline_name: str,
) -> Tuple[List[dict], Dict[str, FitArtifacts]]:

    # 1) target 过滤（每个 target 分开过滤，避免 y 缺失带来的不一致）
    tr = _filter_nonmissing(df_in, train_idx, target)
    va = _filter_nonmissing(df_in, val_idx, target)
    te = _filter_nonmissing(df_in, test_idx, target)

    X_tr = df_in.iloc[tr][feature_cols]
    y_tr = df_in.iloc[tr][target].astype(float).values
    X_va = df_in.iloc[va][feature_cols]
    y_va = df_in.iloc[va][target].astype(float).values
    X_te = df_in.iloc[te][feature_cols]
    y_te = df_in.iloc[te][target].astype(float).values

    # tail 阈值只用 train 算
    q05, q95 = compute_tail_thresholds(y_tr, 0.05, 0.95)

    # 2) baseline fit（RMSE/MSE）
    pipe = build_pipeline(baseline_name)
    t0 = time.time()
    print(f"🚀 Fitting {baseline_name} on target={target} | n_tr={len(y_tr)}")
    pipe.fit(X_tr, y_tr)
    dt = time.time() - t0
    try:
        n_iter = getattr(pipe.named_steps['model'], 'n_iter_', None)
    except Exception:
        n_iter = None
    print(f"✅ Done {baseline_name} | {dt:.1f}s | n_iter={n_iter}")

    yhat_tr = pipe.predict(X_tr)
    yhat_te = pipe.predict(X_te)

    # 温度（按 train 自适应）
    T = compute_temperature(y_tr, CL_TEMP_FACTOR)

    rows = []
    arts = {}

    # ---- Variant A: Baseline ----
    m = evaluate_with_tail(y_te, yhat_te, q05, q95)
    row = {'Target': target, 'Model': baseline_name}
    row.update(m)
    rows.append(row)
    arts[baseline_name] = FitArtifacts(y_true=y_te, y_pred=yhat_te, tail_q05=q05, tail_q95=q95)

    # ---- Variant B: (Baseline)_CL ----
    calib_cl = train_affine_calibrator(yhat_tr, y_tr, T=T, mode='cl')
    yhat_te_cl = calib_cl.a * yhat_te + calib_cl.b

    m = evaluate_with_tail(y_te, yhat_te_cl, q05, q95)
    name_cl = f'{baseline_name}_CL'
    row = {'Target': target, 'Model': name_cl}
    row.update(m)
    rows.append(row)

    diag_p = _approx_test_diag_p(yhat_te_cl, y_te, T=T, batch_size=256, seed=SEED)
    arts[name_cl] = FitArtifacts(
        y_true=y_te,
        y_pred=yhat_te_cl,
        tail_q05=q05,
        tail_q95=q95,
        diag_test=diag_p,
        diag_name='p_i',
        extra={'T': T, 'calib': calib_cl}
    )

    # ---- Variant C: (Baseline)_RMSE_CL ----
    calib_mix = train_affine_calibrator(yhat_tr, y_tr, T=T, mode='rmse_cl', rmse_weight=0.95, cl_weight=0.05)
    yhat_te_mix = calib_mix.a * yhat_te + calib_mix.b

    m = evaluate_with_tail(y_te, yhat_te_mix, q05, q95)
    name_mix = f'{baseline_name}_RMSE_CL'
    row = {'Target': target, 'Model': name_mix}
    row.update(m)
    rows.append(row)

    diag_p2 = _approx_test_diag_p(yhat_te_mix, y_te, T=T, batch_size=256, seed=SEED)
    arts[name_mix] = FitArtifacts(
        y_true=y_te,
        y_pred=yhat_te_mix,
        tail_q05=q05,
        tail_q95=q95,
        diag_test=diag_p2,
        diag_name='p_i',
        extra={'T': T, 'calib': calib_mix}
    )

    return rows, arts


def run_all_models_on_split(df_in, train_idx, val_idx, test_idx, target: str):
    rows_all = []
    arts_all = {}
    for i, bname in enumerate(BASELINE_NAMES, start=1):
        print(f"\n==================== [{i}/{len(BASELINE_NAMES)}] {bname} ====================")
        rows, arts = fit_one_model_three_variants(df_in, train_idx, val_idx, test_idx, target, bname)
        rows_all.extend(rows)
        arts_all.update(arts)
    return rows_all, arts_all

In [11]:
# Split size 打印（只统计 non-missing y）

def pretty_print_split_sizes(df_in, train_idx, val_idx, test_idx, target: str):
    n_train = df_in.iloc[train_idx][target].notna().sum()
    n_val = df_in.iloc[val_idx][target].notna().sum()
    n_test = df_in.iloc[test_idx][target].notna().sum()
    print(f'[{target}] train/val/test (non-missing y): {n_train}/{n_val}/{n_test}')


In [12]:
# 运行两种 split（Random / GroupSplit(grid_id)）并汇总结果

results_rows = []
artifacts_all = {}  # (split_name, target) -> arts

# ---- Split 1: Random ----
train_idx, val_idx, test_idx = make_random_split(df, test_size=0.15, val_size=0.15, seed=SEED)

for target in target_cols:
    pretty_print_split_sizes(df, train_idx, val_idx, test_idx, target)
    rows, arts = run_all_models_on_split(df, train_idx, val_idx, test_idx, target)
    for r in rows:
        r['Split'] = 'Random(70/15/15)'
        results_rows.append(r)
    artifacts_all[('Random(70/15/15)', target)] = arts

# ---- Split 2: GroupSplit(grid_id) ----
if 'grid_id' in df.columns and df['grid_id'].nunique() >= 2:
    train_idx_g, val_idx_g, test_idx_g = make_group_split_by_grid(df, group_col='grid_id', test_size=0.15, val_size=0.15, seed=SEED)
    for target in target_cols:
        pretty_print_split_sizes(df, train_idx_g, val_idx_g, test_idx_g, target)
        rows, arts = run_all_models_on_split(df, train_idx_g, val_idx_g, test_idx_g, target)
        for r in rows:
            r['Split'] = 'GroupSplit(grid_id)'
            results_rows.append(r)
        artifacts_all[('GroupSplit(grid_id)', target)] = arts
else:
    print('⚠️ 未找到可用 grid_id，跳过 GroupSplit(grid_id)。')

results_df = pd.DataFrame(results_rows)
print('Done. Total result rows:', len(results_df))
display(results_df.head(10))


[dVIIRS] train/val/test (non-missing y): 17832/3820/3844

==================== [1/7] XGBoost ====================
🚀 Fitting XGBoost on target=dVIIRS | n_tr=17832
✅ Done XGBoost | 0.9s | n_iter=None
[calib] mode=cl n=17832 epochs=40 batch=128 lr=0.05 | rmse0=2.238322 cl0≈6.038120 (subsampled)
[calib] ep=1/40 step=50 | loss≈4.552043 rmse≈2.763179 cl≈4.552043 | a=2.47252 b=-0.31290 | 0.1s
[calib] ep=1/40 step=100 | loss≈4.518742 rmse≈3.512519 cl≈4.518742 | a=2.78139 b=-0.33541 | 0.1s
[calib] ✅ epoch_end ep=1/40 | loss=5.857002 rmse=5.403583 cl≈5.857002 | a=3.04094 b=-0.48502 | 0.1s
[calib] ep=2/40 step=150 | loss≈4.510149 rmse≈5.625063 cl≈4.510149 | a=3.10391 b=-0.52994 | 0.1s
[calib] ep=2/40 step=200 | loss≈4.499228 rmse≈5.219660 cl≈4.499228 | a=3.05384 b=-0.47385 | 0.2s
[calib] ep=2/40 step=250 | loss≈4.491720 rmse≈5.289176 cl≈4.491720 | a=2.97275 b=-0.45417 | 0.2s
[calib] ✅ epoch_end ep=2/40 | loss=5.861951 rmse=5.424410 cl≈5.861951 | a=3.04827 b=-0.54703 | 0.2s
[calib] ep=3/40 step=30

,Target,Model,All_MAE,All_RMSE,All_R2,All_AbsErr_P95,All_AbsErr_P99,Tail_Rate,Tail_MAE,Tail_RMSE,Tail_R2,Tail_AbsErr_P95,Tail_AbsErr_P99,Split
0,dVIIRS,XGBoost,2.434138,4.332574,0.050075,8.313144,17.960772,0.105359,9.283709,11.398351,0.143734,22.656083,35.398219,Random(70/15/15)
1,dVIIRS,XGBoost_CL,3.565273,6.496370,-1.135693,12.395050,27.215164,0.105359,9.981509,14.124842,-0.314897,29.665650,45.536178,Random(70/15/15)
2,dVIIRS,XGBoost_RMSE_CL,2.502983,4.441382,0.001764,8.473356,18.250166,0.105359,9.174849,11.420684,0.140376,22.709652,36.317506,Random(70/15/15)
3,dVIIRS,Ridge,2.475284,4.426561,0.008415,8.391092,17.521230,0.105359,10.333975,12.215067,0.016632,23.864181,35.334193,Random(70/15/15)
4,dVIIRS,Ridge_CL,3.888343,5.861105,-0.738426,11.747503,21.687710,0.105359,10.248158,13.214137,-0.150806,24.917843,38.571161,Random(70/15/15)
5,dVIIRS,Ridge_RMSE_CL,2.468699,4.424792,0.009207,8.347559,17.548996,0.105359,10.344467,12.221836,0.015542,23.730784,35.205869,Random(70/15/15)
6,dVIIRS,ElasticNet,2.444626,4.445630,-0.000147,8.329953,18.183362,0.105359,10.465397,12.332081,-0.002299,23.036421,33.791832,Random(70/15/15)
7,dVIIRS,ElasticNet_CL,2.898363,4.640751,-0.089867,8.767547,17.762599,0.105359,10.372965,12.343618,-0.004175,22.936817,34.289056,Random(70/15/15)
8,dVIIRS,ElasticNet_RMSE_CL,2.443925,4.445938,-0.000285,8.341107,18.182824,0.105359,10.466813,12.333116,-0.002467,23.057074,33.811372,Random(70/15/15)
9,dVIIRS,SVR,2.505612,4.548892,-0.047149,8.494477,18.547685,0.105359,10.285264,12.403428,-0.013930,23.658845,40.434148,Random(70/15/15)


In [13]:
import pandas as pd

pd.set_option('display.max_rows', 5000)
pd.set_option('display.max_columns', 200)

RESULT1 = results_df.copy()

# pivot：按 split + target 做一个宽表，便于直接对比
metrics_cols = [c for c in RESULT1.columns if c.startswith('All_') or c.startswith('Tail_') or c == 'Tail_Rate']
RESULT2 = RESULT1.pivot_table(index=['Split','Target'], columns='Model', values=metrics_cols)

display(RESULT1.head(20))
print('\nPivot RESULT2 (multi-index columns):')
display(RESULT2.head(10))

# -----------------------------
# ✅ 保存为 Excel（两个 sheet）
# -----------------------------
out_path = "results_XGB.xlsx"

# Excel 更友好：把 RESULT2 的 multi-index columns 拍平
RESULT2_flat = RESULT2.copy()
RESULT2_flat.columns = [f"{metric}__{model}" for (metric, model) in RESULT2_flat.columns.to_list()]
RESULT2_flat = RESULT2_flat.reset_index()

with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    RESULT1.to_excel(writer, sheet_name="RESULT1_long", index=False)
    RESULT2_flat.to_excel(writer, sheet_name="RESULT2_pivot", index=False)

print(f"✅ Saved: {out_path}")



,Target,Model,All_MAE,All_RMSE,All_R2,All_AbsErr_P95,All_AbsErr_P99,Tail_Rate,Tail_MAE,Tail_RMSE,Tail_R2,Tail_AbsErr_P95,Tail_AbsErr_P99,Split
0,dVIIRS,XGBoost,2.434138,4.332574,0.050075,8.313144,17.960772,0.105359,9.283709,11.398351,0.143734,22.656083,35.398219,Random(70/15/15)
1,dVIIRS,XGBoost_CL,3.565273,6.496370,-1.135693,12.395050,27.215164,0.105359,9.981509,14.124842,-0.314897,29.665650,45.536178,Random(70/15/15)
2,dVIIRS,XGBoost_RMSE_CL,2.502983,4.441382,0.001764,8.473356,18.250166,0.105359,9.174849,11.420684,0.140376,22.709652,36.317506,Random(70/15/15)
3,dVIIRS,Ridge,2.475284,4.426561,0.008415,8.391092,17.521230,0.105359,10.333975,12.215067,0.016632,23.864181,35.334193,Random(70/15/15)
4,dVIIRS,Ridge_CL,3.888343,5.861105,-0.738426,11.747503,21.687710,0.105359,10.248158,13.214137,-0.150806,24.917843,38.571161,Random(70/15/15)
5,dVIIRS,Ridge_RMSE_CL,2.468699,4.424792,0.009207,8.347559,17.548996,0.105359,10.344467,12.221836,0.015542,23.730784,35.205869,Random(70/15/15)
6,dVIIRS,ElasticNet,2.444626,4.445630,-0.000147,8.329953,18.183362,0.105359,10.465397,12.332081,-0.002299,23.036421,33.791832,Random(70/15/15)
7,dVIIRS,ElasticNet_CL,2.898363,4.640751,-0.089867,8.767547,17.762599,0.105359,10.372965,12.343618,-0.004175,22.936817,34.289056,Random(70/15/15)
8,dVIIRS,ElasticNet_RMSE_CL,2.443925,4.445938,-0.000285,8.341107,18.182824,0.105359,10.466813,12.333116,-0.002467,23.057074,33.811372,Random(70/15/15)
9,dVIIRS,SVR,2.505612,4.548892,-0.047149,8.494477,18.547685,0.105359,10.285264,12.403428,-0.013930,23.658845,40.434148,Random(70/15/15)



Pivot RESULT2 (multi-index columns):


All_AbsErr_P95                                   \
Model                             ElasticNet ElasticNet_CL ElasticNet_RMSE_CL   
Split               Target                                                      
GroupSplit(grid_id) dVIIRS          8.767010      8.903908           8.769485   
                    dWorldPop      72.782042    148.647823          76.317375   
Random(70/15/15)    dVIIRS          8.329953      8.767547           8.341107   
                    dWorldPop      78.174254    146.818886          84.186044   

                                                                           \
Model                         ExtraTrees ExtraTrees_CL ExtraTrees_RMSE_CL   
Split               Target                                                  
GroupSplit(grid_id) dVIIRS      8.532235      8.526477           8.518925   
                    dWorldPop  68.288788     68.205916          68.301328   
Random(70/15/15)    dVIIRS      8.340090      8.354218           8.339408   
                    dWorldPop  25.658865     25.902496          25.790749   

                                                                               \
Model                                MLP      MLP_CL MLP_RMSE_CL RandomForest   
Split               Target                                                      
GroupSplit(grid_id) dVIIRS      8.700455   14.612969    8.702603     8.400977   
                    dWorldPop  81.935904  105.668612   83.870659    66.209254   
Random(70/15/15)    dVIIRS      8.316272   12.903024    8.358677     7.993396   
                    dWorldPop  40.550128   65.568897   39.859230    36.303788   

                                                                               \
Model                         RandomForest_CL RandomForest_RMSE_CL      Ridge   
Split               Target                                                      
GroupSplit(grid_id) dVIIRS           9.241910             8.700247   8.683639   
                    dWorldPop       69.650387            66.999080  77.251365   
Random(70/15/15)    dVIIRS           8.892441             8.416036   8.391092   
                    dWorldPop       35.698132            34.273601  83.055316   

                                                                    \
Model                            Ridge_CL Ridge_RMSE_CL        SVR   
Split               Target                                           
GroupSplit(grid_id) dVIIRS      11.508242      8.708711   8.730268   
                    dWorldPop  135.943246     77.003798  69.120528   
Random(70/15/15)    dVIIRS      11.747503      8.347559   8.494477   
                    dWorldPop  133.625546     81.211806  64.663161   

                                                                             \
Model                              SVR_CL SVR_RMSE_CL    XGBoost XGBoost_CL   
Split               Target                                                    
GroupSplit(grid_id) dVIIRS      16.935205    8.943560   8.553382  13.411774   
                    dWorldPop  116.360501   74.160173  66.258075  80.717731   
Random(70/15/15)    dVIIRS      14.752623    8.824896   8.313144  12.395050   
                    dWorldPop  103.581654   63.456653  32.376395  49.366668   

                                              All_AbsErr_P99                \
Model                         XGBoost_RMSE_CL     ElasticNet ElasticNet_CL   
Split               Target                                                   
GroupSplit(grid_id) dVIIRS           9.156218      19.742553     19.033001   
                    dWorldPop       67.723662     154.418760    322.433540   
Random(70/15/15)    dVIIRS           8.473356      18.183362     17.762599   
                    dWorldPop       31.537547     185.801739    286.563436   

                                                                            \
Model                         ElasticNet_RMSE_CL  ExtraTrees ExtraTrees_CL   
Split               Target            

✅ Saved: results_XGB.xlsx


In [14]:
# 可视化（更好看一点）：Pred vs True（hexbin + tail overlay）+ Residual + p_i 诊断

plt.rcParams.update({
    'figure.figsize': (6.5, 4.5),
    'axes.grid': True,
    'grid.alpha': 0.2,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
})


def _annot_metrics(ax, y_true, y_pred):
    m = regression_metrics(y_true, y_pred)
    txt = f"RMSE={m['RMSE']:.3f}\nMAE={m['MAE']:.3f}\nR2={m['R2']:.3f}"
    ax.text(0.02, 0.98, txt, transform=ax.transAxes, va='top', ha='left', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.85, linewidth=0.5))


def plot_pred_vs_true(y_true, y_pred, q05=None, q95=None, title='Pred vs True'):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    fig, ax = plt.subplots()

    if (q05 is not None) and (q95 is not None):
        m = tail_mask(y_true, q05, q95)
        # 非 tail 用 hexbin 防止过绘制
        if (~m).sum() > 0:
            ax.hexbin(y_true[~m], y_pred[~m], gridsize=45, mincnt=1, linewidths=0.0)
        # tail 叠加散点
        if m.sum() > 0:
            ax.scatter(y_true[m], y_pred[m], s=18, marker='x', alpha=0.9, label='Tail')
    else:
        ax.scatter(y_true, y_pred, s=8, alpha=0.5)

    mn = float(np.min([np.min(y_true), np.min(y_pred)]))
    mx = float(np.max([np.max(y_true), np.max(y_pred)]))
    ax.plot([mn, mx], [mn, mx], linestyle='--', linewidth=1, alpha=0.8)

    ax.set_xlabel('True')
    ax.set_ylabel('Pred')
    ax.set_title(title)

    _annot_metrics(ax, y_true, y_pred)

    if (q05 is not None) and (q95 is not None):
        ax.legend(loc='lower right', framealpha=0.8)

    fig.tight_layout()
    return fig


def plot_residual_hist(y_true, y_pred, title='Residual distribution'):
    res = (np.asarray(y_pred) - np.asarray(y_true))
    fig, ax = plt.subplots()
    ax.hist(res, bins=60, alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel('Residual (pred - true)')
    ax.set_ylabel('Count')
    fig.tight_layout()
    return fig


def plot_contrastive_diag_p(art: FitArtifacts, title='p_i distribution'):
    if art.diag_test is None or len(art.diag_test) == 0:
        return None
    v = np.asarray(art.diag_test)
    fig, ax = plt.subplots()
    ax.hist(v, bins=50, alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel('p_i')
    ax.set_ylabel('Count')
    fig.tight_layout()
    return fig


def plot_triplet(arts: Dict[str, FitArtifacts], base_name: str, split_name: str, target: str):
    keys = [base_name, f'{base_name}_CL', f'{base_name}_RMSE_CL']
    fig, axes = plt.subplots(1, 3, figsize=(15.8, 4.5))

    for ax, k in zip(axes, keys):
        art = arts[k]
        y_true, y_pred = art.y_true, art.y_pred
        m = tail_mask(y_true, art.tail_q05, art.tail_q95)

        # ---- non-tail: light scatter (density via alpha accumulation) ----
        bulk_color = "#5183C8"   # ink
        tail_color = "#CA7D7D"   # coral
        diag_color = "#12171D"   # cool gray-blue

        # non-tail
        ax.scatter(
            y_true[~m], y_pred[~m],
            s=16, marker='x', alpha=0.4,
            linewidths=0.8,
            rasterized=True,
            c=bulk_color
        )

        # tail
        ax.scatter(
            y_true[m], y_pred[m],
            s=22, marker='x', alpha=0.6,
            linewidths=1.2,
            c=tail_color
        )


        mn = float(np.min([np.min(y_true), np.min(y_pred)]))
        mx = float(np.max([np.max(y_true), np.max(y_pred)]))
        ax.plot([mn, mx], [mn, mx], linestyle='--', linewidth=1, alpha=0.8, color=diag_color)
        ax.set_title(k)
        ax.set_xlabel('True')
        ax.set_ylabel('Pred')
        _annot_metrics(ax, y_true, y_pred)

    fig.suptitle(f'[{split_name}] {target} - {base_name}: Base vs CL vs RMSE_CL', y=1.02, fontsize=13)
    fig.tight_layout()
    return fig


In [15]:
# 输出目录与保存工具
import os, re

OUTDIR = 'paper_figs_multibaselines'
os.makedirs(OUTDIR, exist_ok=True)


def safe(s: str) -> str:
    return re.sub(r"[^a-zA-Z0-9._-]+", "_", s)


def save_fig(fig, name: str):
    path = os.path.join(OUTDIR, name)
    fig.savefig(path, dpi=400, bbox_inches='tight')
    plt.close(fig)
    return path

print('OUTDIR =', OUTDIR)


OUTDIR = paper_figs_multibaselines


In [16]:
# 为每个 split + target + baseline 生成关键图（更利于论文展示）

for split_name, target in [(s,t) for (s,t) in artifacts_all.keys()]:
    arts = artifacts_all[(split_name, target)]

    for base_name in BASELINE_NAMES:
        # 1) 三联图：Base / CL / RMSE_CL
        fig = plot_triplet(arts, base_name, split_name, target)
        save_fig(fig, f"{safe(split_name)}__{safe(target)}__{safe(base_name)}__triplet.png")

        # 2) p_i 诊断（只对 CL 与 RMSE_CL）
        for k in [f'{base_name}_CL', f'{base_name}_RMSE_CL']:
            art = arts[k]
            figp = plot_contrastive_diag_p(art, title=f'[{split_name}] {k} - {target}: p_i')
            if figp is not None:
                save_fig(figp, f"{safe(split_name)}__{safe(target)}__{safe(k)}__pi.png")

print('Saved figures:', OUTDIR)


Saved figures: paper_figs_multibaselines


In [18]:
# 导出结果表到 CSV（方便论文引用）

out_csv = 'transluxpop_multibaselines_cl_results_095_005.csv'
RESULT1.to_csv(out_csv, index=False)
print('Saved:', out_csv)

Saved: transluxpop_multibaselines_cl_results_095_005.csv


## 备注（重要实现选择）

- 你要求“每个模型都做 CL vs 无 CL”，但这里新增的 baseline 里很多是 **sklearn 的非可微/不可自定义 objective** 模型（例如 RF/ExtraTrees/SVR）。
- 为了 **不改 base 模型超参**，同时又能严格使用你给的对比式公式，这里采用统一做法：
  - 先训练 base model（RMSE/MSE）得到预测 \(\hat y\)
  - 再训练一个非常轻量的可微校准层 \(\hat y' = a\hat y + b\)
  - `(Baseline)_CL`：只优化 \(\mathcal{L}_{con}\)
  - `(Baseline)_RMSE_CL`：优化 \(0.6\cdot RMSE_{norm} + 0.4\cdot CL_{norm}\)

这种做法的好处：
- 对所有 baseline 统一、可复现、可比较
- 不会改动原模型超参（超参变动只发生在校准层的训练过程）
- reported errors 的口径完全一致

如果你后续想把 “CL 直接作用到模型训练” 上（例如用 PyTorch 实现可微的 Ridge/MLP），可以在保持评估框架不变的前提下单独替换 `fit_one_model_three_variants` 的内部实现。
